In [0]:
%sql
-- Replace 'managed-iot-data' with a folder name of your choice in your bucket
CREATE CATALOG IF NOT EXISTS iot_catalog_p2
MANAGED LOCATION 's3://iot-anomaly-pipeline-parquet/';

In [0]:
%sql
USE CATALOG iot_catalog_p2;

-- 1. Bronze: For raw, unprocessed JSON events
CREATE SCHEMA IF NOT EXISTS bronze;

-- 2. Silver: For filtered and validated readings
CREATE SCHEMA IF NOT EXISTS silver;

-- 3. Gold: For metrics, aggregates, and anomaly flags
CREATE SCHEMA IF NOT EXISTS gold;

In [0]:
%sql
USE CATALOG iot_catalog_p2;
USE SCHEMA bronze;

In [0]:
%sql
-- 1. Create Sensor Readings Table from existing Parquet files
CREATE TABLE IF NOT EXISTS sensor_readings
USING PARQUET
LOCATION 's3://iot-anomaly-pipeline-parquet/sensor/'
OPTIONS (
  recursiveFileLookup = "true",
  mergeSchema = "true"
);

In [0]:
%sql
CREATE TABLE IF NOT EXISTS health_diagnostics
USING PARQUET
LOCATION 's3://iot-anomaly-pipeline-parquet/health/'
OPTIONS (
  recursiveFileLookup = "true",
  mergeSchema = "true"
);

In [0]:
%sql
CREATE TABLE IF NOT EXISTS operations_table
USING PARQUET
LOCATION 's3://iot-anomaly-pipeline-parquet/operations/'
OPTIONS (
  recursiveFileLookup = "true",
  mergeSchema = "true"
);

In [0]:
%sql
CREATE TABLE IF NOT EXISTS environment_table
USING PARQUET
LOCATION 's3://iot-anomaly-pipeline-parquet/environment/'
OPTIONS (
  recursiveFileLookup = "true",
  mergeSchema = "true"
);

In [0]:
%sql
-- 5. Create Time Events Table
CREATE TABLE IF NOT EXISTS time_events
USING PARQUET
LOCATION 's3://iot-anomaly-pipeline-parquet/time/'
OPTIONS (
  recursiveFileLookup = "true",
  mergeSchema = "true"
);

In [0]:
%sql
USE CATALOG iot_catalog_p2;

In [0]:
%sql
select * from bronze.health_diagnostics;

In [0]:
from pyspark.sql.functions import col, count, when
from pyspark.sql.types import StringType

# List of your bronze tables
bronze_tables = [
    "sensor_readings", 
    "health_diagnostics", 
    "operations_table", 
    "environment_table", 
    "time_events"
]

for table_name in bronze_tables:
    print(f"--- Null Count Report for: {table_name} ---")
    
    # Load the table from the bronze schema
    df = spark.table(f"iot_catalog_p2.bronze.{table_name}")
    
    # Create an aggregation expression for all columns
    null_exprs = []
    for c in df.columns:
        condition = col(c).isNull()
        if isinstance(df.schema[c].dataType, StringType):
            condition = condition | (col(c) == "" ) | (col(c) == "null")
        null_exprs.append(count(when(condition, c)).alias(c))
    
    null_counts = df.select(null_exprs)
    
    # Display the result
    null_counts.show()